# BA576 Group 6 Project Workbook
## Predicting chess puzzle difficulty ratings

**Research question:** Can we predict a Lichess chess puzzle's numeric difficulty rating from the position, the known solution, and tactical features available when a puzzle is created?

**Team:** Jonathan Sze, Nawaf Alsheikh, Alexander Lee, Aaron Cheung

This notebook is a collaborative starter workbook. It loads the first 30,000 rows from the Lichess puzzle source, audits the raw data, creates interpretable features, visualizes initial relationships, and establishes a grouped baseline. Verify and adapt every result before using it in the project presentation.

## Project decisions to confirm

1. **Outcome:** `Rating`, a numeric Lichess puzzle difficulty score.
2. **Potential leakage:** solution derived features are included only if the use case is rating a puzzle after its known solution is available. If the use case is rating a position before a solution is available, remove those features.
3. **Evaluation:** split by source game, not only by row, so puzzles from the same game do not appear in both train and test sets.
4. **Excluded fields:** `Popularity`, `NbPlays`, and `RatingDeviation` are post creation or rating system fields and should not be used as predictors without a clear justification.

The BA576 syllabus requires citation of external resources, including AI tools. Add any required disclosure to the final project materials.

In [ ]:
# Core imports
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 576
DATA_FILENAME = 'Lichess_Puzzle_First_30000.csv'

In [ ]:
# Load the first 30,000 rows from Google Drive or upload a source CSV in Colab
ROW_LIMIT = 30_000
def find_data_file():
    filenames = [
        'Lichess_Puzzle_First_30000.csv',
        'Lichess_Puzzle_30000.csv',
        'lichess_db_puzzle.csv',
        'Lichess_Puzzle_Pilot_5000.csv',
    ]
    roots = [
        Path('.'),
        Path('/content'),
        Path('/content/drive/MyDrive/BA576 group Project'),
        Path('/content/drive/MyDrive/BA576 Group Project'),
        Path('/content/drive/MyDrive'),
    ]
    candidates = [root / name for root in roots for name in filenames]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import drive, files
        drive.mount('/content/drive')
        for candidate in candidates:
            if candidate.exists():
                return candidate
        print('Upload a Lichess puzzle CSV with at least 30,000 rows when prompted.')
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        return Path(uploaded_name)
    except ImportError:
        raise FileNotFoundError(
            'Could not find a Lichess puzzle CSV with at least 30,000 rows. ' 
            'Set the path manually or run this cell in Colab.'
        )

data_path = find_data_file()
df_raw = pd.read_csv(data_path, nrows=ROW_LIMIT)
if len(df_raw) < ROW_LIMIT:
    raise ValueError(
        f'{data_path} contains only {len(df_raw):,} rows. ' 
        f'This workbook requires at least {ROW_LIMIT:,} rows.'
    )
print(f'Loaded the first {len(df_raw):,} rows from {data_path}')
df_raw.head()

In [ ]:
# Raw data audit
audit = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing': df_raw.isna().sum(),
    'missing_pct': (df_raw.isna().mean() * 100).round(2),
    'unique_values': df_raw.nunique(),
})
print('Shape:', df_raw.shape)
print('Duplicate PuzzleId rows:', df_raw['PuzzleId'].duplicated().sum())
display(audit)
display(df_raw.describe(include='all').T)

In [ ]:
# Feature engineering from FEN, solution moves, and tactical themes
# The recorded solution starts with the solver's move in the Lichess puzzle format.
try:
    import chess
except ImportError:
    %pip -q install python-chess
    import chess

PIECE_VALUES = {'P': 1, 'N': 3, 'B': 3, 'R': 5, 'Q': 9, 'K': 0}

def fen_features(fen):
    board_part, active_color, _, _, _, fullmove = fen.split()
    counts = {piece: 0 for piece in 'PNBRQKpnbrqk'}
    for char in board_part:
        if char in counts:
            counts[char] += 1
    board = chess.Board(fen)
    solver = active_color == 'w'
    features = {
        'solver_pawn_count': counts['P'] if solver else counts['p'],
        'opponent_pawn_count': counts['p'] if solver else counts['P'],
        'total_knights': counts['N'] + counts['n'],
        'total_bishops': counts['B'] + counts['b'],
        'total_rooks': counts['R'] + counts['r'],
        'total_queens': counts['Q'] + counts['q'],
        'material_balance': sum(PIECE_VALUES[p] * counts[p] for p in 'PNBRQK') - sum(PIECE_VALUES[p] * counts[p.lower()] for p in 'PNBRQK'),
        'fullmove_number': int(fullmove),
        'solver_is_white': int(solver),
        'in_check': int(board.is_check()),
        'legal_move_count': board.legal_moves.count(),
    }
    if not solver:
        features['material_balance'] *= -1
    return features

def solution_features(fen, moves):
    board = chess.Board(fen)
    tokens = str(moves).split()
    solver_move_count = (len(tokens) + 1) // 2
    captures = checks = solver_captures = solver_checks = 0
    first_capture = first_check = 0
    for index, token in enumerate(tokens):
        move = chess.Move.from_uci(token)
        is_solver_move = index % 2 == 0
        is_capture = int(board.is_capture(move))
        board.push(move)
        gives_check = int(board.is_check())
        captures += is_capture
        checks += gives_check
        if is_solver_move:
            solver_captures += is_capture
            solver_checks += gives_check
            if index == 0:
                first_capture = is_capture
                first_check = gives_check
    return {
        'solver_move_count': solver_move_count,
        'total_captures': captures,
        'total_checks': checks,
        'solver_captures': solver_captures,
        'solver_checks': solver_checks,
        'first_solver_move_capture': first_capture,
        'first_solver_move_check': first_check,
    }

def theme_features(themes):
    tags = set(str(themes).split())
    return {f'theme_{tag}': int(tag in tags) for tag in ['fork', 'pin', 'sacrifice']}

fen_df = pd.DataFrame(df_raw['FEN'].map(fen_features).tolist())
solution_df = pd.DataFrame([solution_features(fen, moves) for fen, moves in zip(df_raw['FEN'], df_raw['Moves'])])
theme_df = pd.DataFrame(df_raw['Themes'].map(theme_features).tolist())
df = pd.concat([df_raw[['PuzzleId', 'Rating', 'GameUrl']], fen_df, solution_df, theme_df], axis=1)
df['source_game_id'] = df['GameUrl'].str.extract(r'lichess\.org/([^/#]+)', expand=False)
df.head()

In [ ]:
# Initial descriptive analysis
print(df[['Rating']].describe().round(2))

plot_df = df.copy()
plot_df['solver_move_bin'] = plot_df['solver_move_count'].clip(upper=5).astype(str)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=plot_df, x='Rating', bins=30, ax=axes[0])
axes[0].set_title('Distribution of puzzle ratings')
axes[0].set_xlabel('Lichess rating')
sns.boxplot(data=plot_df, x='solver_move_bin', y='Rating', ax=axes[1])
axes[1].set_title('Rating by solver move count')
axes[1].set_xlabel('Solver moves, 5 means 5 or more')
plt.tight_layout()
plt.show()

theme_means = df.groupby('theme_fork')['Rating'].mean().rename({0: 'No fork', 1: 'Fork'})
display(theme_means.to_frame('mean_rating').round(1))

In [ ]:
# Grouped train and test split plus simple baselines
feature_cols = [
    'solver_pawn_count', 'opponent_pawn_count', 'total_knights', 'total_bishops',
    'total_rooks', 'total_queens', 'material_balance', 'fullmove_number',
    'solver_is_white', 'in_check', 'legal_move_count', 'solver_move_count',
    'total_captures', 'total_checks',
    'first_solver_move_capture', 'first_solver_move_check',
    'theme_fork', 'theme_pin', 'theme_sacrifice',
]
X = df[feature_cols]
y = df['Rating']
groups = df['source_game_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f'Train rows: {len(train_idx):,}; test rows: {len(test_idx):,}')
print('Shared source games:', len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

models = {
    'mean baseline': DummyRegressor(strategy='mean'),
    'ridge baseline': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10.0)),
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'R2': r2_score(y_test, pred),
    })

pd.DataFrame(results).sort_values('MAE').round(2)

## Next work items

- Confirm whether solution derived predictors are allowed by the intended prediction use case.
- Confirm that the first 30,000 rows are an acceptable working sample and consider a random or stratified sample for sensitivity analysis.
- Check whether the source game grouping is sufficient to prevent duplicate game leakage.
- Add the methods taught in class as they are covered: linear regression, regularization, trees, bagging, random forests, boosting, and neural networks where appropriate.
- Compare models using the same grouped holdout and report MAE, RMSE, and R squared.
- Inspect residuals and failure cases, then document the biggest modeling challenge.
- Keep a short decision log in the repository so collaborators can see why features and models were included or removed.